In [1]:
import numpy as np

np.random.seed(42)
# 회귀문제로 XGBoost 만들기!
X = np.random.rand(100,1)*10
y = 2*X.squeeze() + np.random.randn(100)*2

split_index = int(0.8*len(X))
X_train, X_test = X[:split_index], X[split_index:]
y_train, y_test = y[:split_index], y[split_index:]

In [10]:
##Xgboost 함수 만들기
def calculate_gradient(y, y_pred):
    """그래디언트(오차의 음의 기울기)를 계산."""
    return y - y_pred  # 예측값과 실제값의 차이 계산

def calculate_hessian(y, y_pred):
    """헤시안(2차 도함수)을 계산. 회귀의 경우 상수값."""
    return np.ones_like(y)  # 제곱 오차의 경우 헤시안은 상수 (1)

def split_data(X, y, gradient, hessian):
    """데이터를 최적의 분할 기준으로 분리."""
    best_split = None  # 최적의 분할 정보를 저장할 변수
    best_gain = -np.inf  # 가장 높은 이득 값 초기화

    for feature_index in range(X.shape[1]):  # 각 특성에 대해 반복
        thresholds = np.unique(X[:, feature_index])  # 고유 임계값을 추출

        for threshold in thresholds:  # 각 임계값에 대해 반복
            left_mask = X[:, feature_index] <= threshold  # 임계값 이하 데이터 선택
            right_mask = ~left_mask  # 임계값 초과 데이터 선택

            if sum(left_mask) == 0 or sum(right_mask) == 0:  # 빈 노드 방지
                continue

            # 왼쪽과 오른쪽 노드의 그래디언트와 헤시안 계산
            g_left, g_right = gradient[left_mask], gradient[right_mask]
            h_left, h_right = hessian[left_mask], hessian[right_mask]

            # 이득(gain) 계산
            gain = (
                (g_left.sum() ** 2) / (h_left.sum() + 1e-6)  # 왼쪽 노드의 이득
                + (g_right.sum() ** 2) / (h_right.sum() + 1e-6)  # 오른쪽 노드의 이득
                - (gradient.sum() ** 2) / (hessian.sum() + 1e-6)  # 전체 이득
            )

            # 가장 높은 이득을 가진 분할 찾기
            if gain > best_gain:
                best_gain = gain
                best_split = {
                    "feature_index": feature_index,  # 분할에 사용된 특성 인덱스
                    "threshold": threshold,  # 분할 임계값
                    "gain": gain,  # 이득 값
                }

    return best_split

def create_leaf(gradient, hessian):
    """리프 노드 생성 (최적 가중치 계산)."""
    return -gradient.sum() / (hessian.sum() + 1e-6)  # 가중치 계산

# 트리 구조 정의
class Tree:
    def __init__(self, max_depth):
        self.max_depth = max_depth  # 트리의 최대 깊이 설정
        self.tree = {}  # 트리 구조 저장 변수

    def fit(self, X, y, gradient, hessian, depth=0):
        # 최대 깊이 도달하거나 샘플이 1개 이하인 경우 리프 노드 생성
        if depth >= self.max_depth or len(y) <= 1:
            return create_leaf(gradient, hessian)

        # 최적 분할 찾기
        split = split_data(X, y, gradient, hessian)

        if not split:  # 분할 불가능한 경우 리프 노드 생성
            return create_leaf(gradient, hessian)

        left_mask = X[:, split["feature_index"]] <= split["threshold"]  # 왼쪽 노드 데이터
        right_mask = ~left_mask  # 오른쪽 노드 데이터

        # 재귀적으로 왼쪽과 오른쪽 서브트리 생성
        self.tree = {
            "split": split,  # 분할 정보 저장
            "left": self.fit(X[left_mask], y[left_mask], gradient[left_mask], hessian[left_mask], depth + 1),
            "right": self.fit(X[right_mask], y[right_mask], gradient[right_mask], hessian[right_mask], depth + 1),
        }
        return self.tree

    def predict_row(self, row):
        """하나의 샘플에 대해 예측."""
        node = self.tree  # 트리의 루트 노드
        while isinstance(node, dict):  # 리프 노드에 도달할 때까지 반복
            if row[node["split"]["feature_index"]] <= node["split"]["threshold"]:
                node = node["left"]  # 왼쪽 서브트리로 이동
            else:
                node = node["right"]  # 오른쪽 서브트리로 이동
        return node  # 리프 노드 값 반환

    def predict(self, X):
        """모든 샘플에 대해 예측."""
        return np.array([self.predict_row(row) for row in X])  # 각 샘플에 대해 예측

# 그레디언트 부스팅 회귀 모델
class XGBoost:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators  # 트리 개수 설정
        self.learning_rate = learning_rate  # 학습률 설정
        self.max_depth = max_depth  # 트리 최대 깊이 설정
        self.trees = []  # 학습된 트리 저장

    def fit(self, X, y):
        y_pred = np.zeros_like(y)  # 초기 예측값 (0으로 초기화)

        for _ in range(self.n_estimators):  # 각 트리에 대해 반복
            gradient = calculate_gradient(y, y_pred)  # 그래디언트 계산
            hessian = calculate_hessian(y, y_pred)  # 헤시안 계산

            tree = Tree(self.max_depth)  # 새 트리 생성
            tree.fit(X, y, gradient, hessian)  # 트리 학습
            self.trees.append(tree)  # 학습된 트리 저장

            # 새로운 트리의 예측값으로 예측 업데이트
            y_pred += self.learning_rate * tree.predict(X)

    def predict(self, X):
        y_pred = np.zeros(X.shape[0])  # 초기 예측값 (0으로 초기화)
        for tree in self.trees:  # 각 트리의 예측값 누적
            y_pred += self.learning_rate * tree.predict(X)
        return y_pred

# 모델 학습 및 평가
model = XGBoost(n_estimators=10, learning_rate=0.1, max_depth=3)  # 모델 초기화
model.fit(X_train, y_train)  # 모델 학습

y_pred = model.predict(X_test)  # 테스트 데이터 예측

# 평균 제곱 오차 계산
mse = np.mean((y_test - y_pred) ** 2)  # MSE 계산
print("Mean Squared Error:", mse)  # 결과 출력


Mean Squared Error: 832.2083935617218
